# Champs3D — post-treatment fields, 3D

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).

Merged form: the post-treatment fields that cannot be checked in a purely 2D
form are validated here, so `Champs2D` stays 2D:

| Dataset | Field | Why 3D is required |
|---|---|---|
| `champs3d_gradient` | `courant_magnetique` | the 2D branch returns the single out-of-plane scalar $J_z$; the **vector** $J$ needs 3D |
| `champs3d_transport` | `helicite_cinetique` | $h_k=u\cdot(\nabla\times u)$ is identically zero in 2D (the vorticity is out of plane, $u$ is in plane) |

The two datasets are kept separate for the same reason as in `Champs2D`: the
gradient field needs $\eta=0$ (a linear $b$ is an exact equilibrium), while the
kinetic helicity needs a non-uniform velocity. Each keeps its own 3-level
mesh-convergence ladder.

`champs3d_transport` is also the **scalar-model** (`nb_composantes 1`) home of
`energie_magnetique` and `pulsation_cyclotronique`, whose vector branch is
checked in `Champs2D`: at `nb_composantes 1` the component loop runs at its
degenerate end, so a field reader that hard-coded 2 or 3 components would read
out of bounds.

## Introduction

Validation of the **post-treatment fields** of the standalone magnetic
(induction) equation solved by TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

The fields are created lazily by
`Convection_Diffusion_Magnetic_field::creer_champ()` and evaluated by
`Champ_Fonc_MHD` (`src/Champ_Fonc_MHD.{h,cpp}`) on top of the pure arithmetic
kernels of `src/MHD_posttraitement_kernel.h`.

### `champs3d_gradient` — electric current density, full vector

Exactly linear $b=(y,z,x)$:

$$\nabla\times b = (-1,-1,-1), \qquad
J = \frac{\nabla\times b}{\mu_0} = (-0.5,-0.5,-0.5) \quad (\mu_0 = 2)$$

| # | Quantity | Formula | Expected |
|---|---|---|---|
| 1 | current density | $J=(\nabla\times b)/\mu_0$ | $(-0.5,-0.5,-0.5)$ |

$\mu_0=2$ is non-default, so a formula that ignores the `mu0` keyword is
caught; $\eta=0$ and no velocity make the linear field an exact equilibrium.

### `champs3d_transport` — kinetic helicity, three-component curl

Exactly linear velocity $u=(y,z,x)$:

$$\nabla\times u = (-1,-1,-1), \qquad
h_k = u\cdot(\nabla\times u) = -(x+y+z)$$

| # | Quantity | Formula | Expected |
|---|---|---|---|
| 1 | kinetic helicity | $h_k=u\cdot(\nabla\times u)$ | $-(x+y+z)$ |

### `champs3d_transport` (continued) — scalar-model branch of the $|b|^2$ fields

The same dataset has `nb_composantes 1`, so `energie_magnetique` and
`pulsation_cyclotronique` are probed there too. Both read $b$ only through
$|b|^2$; with uniform $b=1$, $\mu_0=2$ and $q/m=2$:

| # | Quantity | Formula | Expected |
|---|---|---|---|
| 2 | magnetic energy (scalar b) | $e_m=|b|^2/(2\mu_0)$ | $0.25$ |
| 3 | cyclotron pulsation (scalar b) | $\omega_c=(q/m)|b|$ | $2$ |

The reference is position-dependent, so the helicity above is evaluated at the
probe's effective cell-centre position, read back from the `.son` header.

The tolerance is $10^{-8}$ rather than machine precision: the kernel's element
velocity gradient (`grad_u`) is biased in the cells adjacent to a wall, and the
baseline probe sits in such a cell (see `docs/content/dev_corner/kernel_reuse.md`).

In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()

## Helpers: read a TRUST probe (`.son`) file

Each dataset runs in its own `build/<dataset>/<scale>/` directory (the merged
form layout), so the readers take the build subdirectory and the case name
explicitly. `probe_pos()` recovers the **effective** probe position from the
`.son` header: the probe is snapped to the nearest cell centre, and on a
refined mesh the nearest centre is not the requested point.

In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son_dir(subdir, case_name, sonde):
    """(time, values) of probe `sonde` of case `case_name` in build/<subdir>/."""
    path = f"{BUILD}/{subdir}/{case_name}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_scalar_dir(subdir, case_name, sonde):
    """Final value of a scalar post-treatment field at the probe."""
    t, v = read_son_dir(subdir, case_name, sonde)
    return float(v[-1])

def read_vector_dir(subdir, case_name, sonde):
    """Final value of every component of a vector post-treatment field."""
    path = f"{BUILD}/{subdir}/{case_name}_{sonde}.son"
    son = SonFile(path, None)
    return np.array([float(np.asarray(son.getValues(e)[1], dtype=float)[-1])
                     for e in son.getEntries()])

def probe_pos(path, dim=2):
    """Effective probe position (snapped to a cell centre), from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                out = [float(toks[toks.index("x") + 2]),
                       float(toks[toks.index("y") + 2])]
                if dim == 3:
                    out.append(float(toks[toks.index("z") + 2]))
                return tuple(out)
    raise RuntimeError(f"no probe position header in {path}")

print("BUILD =", BUILD)

## Run

Two 3D datasets (`Pave` + `Polyedriser dom`, PolyMAC_MPFA on polyhedra), each
with the shared refinement ladder (`coarse` = the baseline `5x5x5` mesh,
`standard` = `9x9x9`, `fine` = `17x17x17`), each scale in its own
`build/<dataset>/<scale>/` directory. Runs on **2 processors** (`nbProcs=2` +
`case.partition()`).

> **Historical note.** The pre-merge 3D gradient form pinned its `fine` (17³)
> level to a single rank because of a layout-dependent kernel `SIGFPE`
> (`docs/content/dev_corner/kernel_reuse.md` §18.2). With this dataset and case name all three
> levels have been verified stable on 2 ranks, so no rank pin is applied here.

In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder, per-dataset subdirectories) ------
# Each dataset src/<name>.data carries $nx/$ny/$nz node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<name>/<scale>/:
#   coarse   = the baseline mesh (5x5x5)
#   standard = cells x2          (9x9x9)
#   fine     = cells x4          (17x17x17)
# Each scale runs in its own build/<name>/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
LEVELS = (("coarse", 5), ("standard", 9), ("fine", 17))
cases = {}
for base in ("champs3d_gradient", "champs3d_transport"):
    cases[base] = {}
    for lvl, n in LEVELS:
        cases[base][lvl] = run.addCaseFromTemplate(
            f"{base}.data", f"{base}/{lvl}", dic={"nx": n, "ny": n, "nz": n},
            nbProcs=NB_PROCS, excluNR=True)
        cases[base][lvl].partition()
run.printCases()
run.runCases()

## Checks — `champs3d_gradient` (vector current density)

In [ ]:
BASE = "champs3d_gradient"
CN = cases[BASE]["coarse"].dataFileName_
SUB = f"{BASE}/coarse"

J = read_vector_dir(SUB, CN, "SONDE_J")
exp_J = np.array([-0.5, -0.5, -0.5])

print(f"courant_magnetique : {J}   expected {exp_J}")
assert np.max(np.abs(J - exp_J)) < 1e-12, f"courant_magnetique wrong: {J}"
print("OK: courant_magnetique = curl(b) / mu0, full 3D vector branch.")

## Checks — `champs3d_transport` (kinetic helicity + scalar-model $|b|^2$ fields)

In [ ]:
BASE = "champs3d_transport"
CN = cases[BASE]["coarse"].dataFileName_
SUB = f"{BASE}/coarse"

hk  = read_scalar_dir(SUB, CN, "SONDE_HK")
em  = read_scalar_dir(SUB, CN, "SONDE_EM")
om  = read_scalar_dir(SUB, CN, "SONDE_OMEGA")
xyz = probe_pos(f"{BUILD}/{SUB}/{CN}_SONDE_HK.son", dim=3)

exp_hk = -(xyz[0] + xyz[1] + xyz[2])
exp_em = 1.0 / (2 * 2.0)   # |b| = 1 (scalar model), mu0 = 2
exp_om = 2.0 * 1.0         # q/m = 2, |b| = 1

print(f"probe                          : {xyz}")
print(f"helicite_cinetique             : {hk:.15g}   expected {exp_hk:.15g}")
print(f"energie_magnetique (scalar b)  : {em:.15g}   expected {exp_em:.15g}")
print(f"pulsation_cyclotronique (scal.): {om:.15g}   expected {exp_om:.15g}")

assert abs(hk - exp_hk) < 1e-8, f"helicite_cinetique wrong: {hk}"
assert abs(em - exp_em) < 1e-12, f"scalar energie_magnetique wrong: {em}"
assert abs(om - exp_om) < 1e-12, f"scalar pulsation_cyclotronique wrong: {om}"
print("OK: helicite_cinetique = u . curl(u), three-component curl branch;")
print("OK: scalar (nb_composantes 1) branch of the |b|^2 fields.")

## Mesh convergence

Both references are identities of the discrete operators, so the deviation must
stay at machine precision (current, position-independent) or at the documented
`grad_u` wall-cell bias ($10^{-8}$, kinetic helicity, position-dependent) at
**every** mesh level. No convergence order is reported — only that the identity
does not degrade under refinement.

Note that the probe is snapped to the nearest cell centre and the snapped
coordinates change with the mesh (`0.375` at `5x5x5`, `0.3125` at `9x9x9`,
`0.34375` at `17x17x17`), so the helicity reference is recomputed per level
from the `.son` header.

In [ ]:
def err_gradient(sub, cn):
    J = read_vector_dir(sub, cn, "SONDE_J")
    return float(np.max(np.abs(J - np.array([-0.5, -0.5, -0.5]))))

def err_transport(sub, cn):
    hk = read_scalar_dir(sub, cn, "SONDE_HK")
    xyz = probe_pos(f"{BUILD}/{sub}/{cn}_SONDE_HK.son", dim=3)
    return max(
        abs(hk - (-(xyz[0] + xyz[1] + xyz[2]))),
        abs(read_scalar_dir(sub, cn, "SONDE_EM") - 1.0 / (2 * 2.0)),
        abs(read_scalar_dir(sub, cn, "SONDE_OMEGA") - 2.0 * 1.0),
    )

TOL = {"champs3d_gradient": 1e-12, "champs3d_transport": 1e-8}

print(f"{'dataset':<20} {'level':<9} {'nodes':>9}  {'|err|':>12}")
for base, errf in (("champs3d_gradient", err_gradient),
                   ("champs3d_transport", err_transport)):
    for lvl, n in LEVELS:
        e = errf(f"{base}/{lvl}", cases[base][lvl].dataFileName_)
        print(f"{base:<20} {lvl:<9} {n:>3}x{n:<5}  {e:12.3e}")
        assert e < TOL[base], f"{base} identity broken at {lvl}: {e:.3e}"
print("OK: every analytic identity holds at every mesh level.")

## Computer Performance

In [ ]:
run.tablePerf()